# 配套实践 14-02：比较像素、状态与潜空间预测

本练习把一维物体位置渲染成 16×16 光点图像。结构化状态只需一个位置数值，像素表示需要 256 个数值；我们用 PCA 把图像压到低维 latent，再拟合动作条件线性 latent dynamics。实验检查压缩保留了什么，以及忽略动作会怎样破坏未来图像 rollout。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/14-action-conditioned-world-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 渲染光点、计算 PCA 与拟合线性 latent dynamics
import matplotlib.pyplot as plt  # 绘制重建、一步误差和多步未来
np.random.seed(142)  # 固定状态动作数据与测试序列
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 同一物理状态的三种表示

物理状态是区间 [-0.8,0.8] 内的一个位置；像素表示把它渲染成横向移动的 Gaussian 光点。PCA latent 不使用动作标签，只寻找训练图像中变化最大的低维方向。下面比较不同 latent 维度的像素重建。

In [ ]:
image_size = 16  # 设置方形观测图像的边长
grid_y, grid_x = np.mgrid[0:image_size, 0:image_size]  # 建立每个像素的二维坐标网格
def render_position(position):  # 定义把一维物体状态渲染为光点图像的函数
    center_x = 2.0 + (position + 0.8) / 1.6 * 11.0  # 把物理位置线性映射到图像内部横坐标
    center_y = 7.5  # 让物体始终位于图像垂直中心
    image = np.exp(-((grid_x - center_x) ** 2 + (grid_y - center_y) ** 2) / (2.0 * 1.15 ** 2))  # 使用二维 Gaussian 渲染柔和光点
    return image.astype(np.float64)  # 返回双精度像素数组供线性代数使用
sample_count = 1600  # 设置用于拟合表示和动力学的转移数量
current_positions = np.random.uniform(-0.8, 0.8, sample_count)  # 随机采样当前结构化位置状态
actions = np.random.uniform(-0.12, 0.12, sample_count)  # 随机采样每步水平位移动作
next_positions = np.clip(current_positions + actions, -0.8, 0.8)  # 根据动作得到带边界裁剪的下一位置
current_images = np.stack([render_position(position) for position in current_positions])  # 把当前状态渲染成批量像素观测
next_images = np.stack([render_position(position) for position in next_positions])  # 把下一状态渲染成未来像素标签
all_flat_images = np.concatenate([current_images, next_images], axis=0).reshape(-1, image_size ** 2)  # 合并当前与未来图像并展平为像素向量
pixel_mean = all_flat_images.mean(axis=0, keepdims=True)  # 计算训练像素均值用于 PCA 中心化
_, singular_values, right_vectors = np.linalg.svd(all_flat_images - pixel_mean, full_matrices=False)  # 使用奇异值分解得到主变化方向
example_image = render_position(0.37)  # 选择一个未特别对齐训练索引的位置作为重建示例
fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.8))  # 创建原图与三个 latent 维度的重建图
axes[0].imshow(example_image, cmap="viridis", vmin=0.0, vmax=1.0)  # 显示包含二百五十六个像素值的原观测
axes[0].set_title("Pixels: 256D")  # 标注原始像素维度
for axis, latent_dimension in zip(axes[1:], [2, 4, 8]):  # 依次使用二、四、八维 PCA latent 重建
    basis = right_vectors[:latent_dimension]  # 取出指定数量的主要像素变化方向
    latent = (example_image.reshape(1, -1) - pixel_mean) @ basis.T  # 把示例图像编码为低维 latent
    reconstruction = pixel_mean + latent @ basis  # 把 latent 线性解码回像素空间
    axis.imshow(reconstruction.reshape(image_size, image_size), cmap="viridis", vmin=0.0, vmax=1.0)  # 显示当前维度的重建结果
    axis.set_title(f"PCA latent: {latent_dimension}D")  # 标注当前 latent 维度
for axis in axes:  # 统一隐藏四幅图不承载物理意义的像素刻度
    axis.axis("off")  # 只保留光点位置与形状
fig.suptitle("State 1D, pixels 256D, latent keeps dominant visual change")  # 强调三种表示的信息与维度差异
fig.tight_layout()  # 调整重建图间距
plt.show()  # 显示不同潜维度保留的像素信息

**怎样理解结果：** 结构化状态只需位置一个数，像素图却使用 256 维描述同一物体。2 维 latent 已能粗略表达横向位置，但形状和强度失真；维度增加后重建更清晰。重建好只说明 PCA 保留主要像素变化，还需检查这些维度能否在动作条件下预测。

## 2. 在 latent 中拟合有动作和无动作的转移

固定使用 8 维 latent。动作条件模型拟合 latent 残差是当前 latent 与动作的线性函数；无动作模型只能根据当前 latent 预测平均变化。然后在独立测试转移上比较 latent MSE 和解码后的光点位置误差。

In [ ]:
latent_dimension = 8  # 选择能较好重建光点的八维 PCA 表示
latent_basis = right_vectors[:latent_dimension]  # 保存固定的八个像素主方向
def encode_images(images):  # 定义从批量图像到 PCA latent 的编码函数
    flat_images = images.reshape(len(images), -1)  # 把每幅二维图像展平成像素向量
    return (flat_images - pixel_mean) @ latent_basis.T  # 中心化后投影到八维 latent
def decode_latents(latents):  # 定义从 PCA latent 到批量图像的解码函数
    flat_images = pixel_mean + latents @ latent_basis  # 用主方向线性组合恢复像素向量
    return flat_images.reshape(-1, image_size, image_size)  # 把向量恢复为方形图像批量
current_latents = encode_images(current_images)  # 编码全部当前像素观测
next_latents = encode_images(next_images)  # 编码全部下一时刻像素标签
latent_residuals = next_latents - current_latents  # 计算动作导致的潜表示变化
action_design = np.concatenate([current_latents, actions[:, None], np.ones((sample_count, 1))], axis=1)  # 为动作条件线性回归组织 latent、动作和偏置
action_weights = np.linalg.lstsq(action_design, latent_residuals, rcond=None)[0]  # 用最小二乘拟合动作条件 latent 残差
state_design = np.concatenate([current_latents, np.ones((sample_count, 1))], axis=1)  # 为无动作基线只组织当前 latent 和偏置
state_weights = np.linalg.lstsq(state_design, latent_residuals, rcond=None)[0]  # 拟合忽略动作时的平均 latent 变化
test_count = 500  # 设置独立一步测试转移数量
test_positions = np.random.uniform(-0.75, 0.75, test_count)  # 采样避开边界的大部分测试位置
test_actions = np.random.uniform(-0.1, 0.1, test_count)  # 采样训练范围内的测试动作
test_next_positions = np.clip(test_positions + test_actions, -0.8, 0.8)  # 计算真实下一结构化位置
test_current_images = np.stack([render_position(position) for position in test_positions])  # 渲染测试当前像素
test_next_images = np.stack([render_position(position) for position in test_next_positions])  # 渲染测试未来像素
test_current_latents = encode_images(test_current_images)  # 编码测试当前观测
test_next_latents = encode_images(test_next_images)  # 编码测试未来标签
test_action_design = np.concatenate([test_current_latents, test_actions[:, None], np.ones((test_count, 1))], axis=1)  # 组织动作条件测试输入
test_state_design = np.concatenate([test_current_latents, np.ones((test_count, 1))], axis=1)  # 组织无动作测试输入
predicted_action_latents = test_current_latents + test_action_design @ action_weights  # 预测动作条件下一 latent
predicted_state_latents = test_current_latents + test_state_design @ state_weights  # 预测忽略动作的下一 latent
def image_centers(images):  # 定义从光点图像估计横向中心的可解释测量函数
    positive_images = np.clip(images, 0.0, None)  # 避免 PCA 重建中的微小负像素影响质心
    horizontal_mass = positive_images.sum(axis=1)  # 沿图像纵向求和得到横向亮度分布
    pixel_centers = (horizontal_mass * np.arange(image_size)[None, :]).sum(axis=1) / (horizontal_mass.sum(axis=1) + 1e-8)  # 计算每幅图的亮度横向质心
    return (pixel_centers - 2.0) / 11.0 * 1.6 - 0.8  # 把像素质心映射回物理位置坐标
action_decoded = decode_latents(predicted_action_latents)  # 把动作条件 latent 预测解码为未来图像
state_decoded = decode_latents(predicted_state_latents)  # 把无动作 latent 预测解码为未来图像
latent_errors = [np.mean((predicted_action_latents - test_next_latents) ** 2), np.mean((predicted_state_latents - test_next_latents) ** 2)]  # 计算两种模型的一步 latent MSE
position_errors = [np.mean(np.abs(image_centers(action_decoded) - test_next_positions)), np.mean(np.abs(image_centers(state_decoded) - test_next_positions))]  # 计算解码后的一步物理位置误差
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.5))  # 创建 latent 与物理位置两类误差柱状图
model_names = ["Latent + action", "Latent only"]  # 定义两种潜空间动力学名称
axes[0].bar(model_names, latent_errors, color=["#2563eb", "#94a3b8"])  # 比较潜表示预测 MSE
axes[0].set(title="One-step latent error", ylabel="MSE")  # 标注潜空间误差含义
axes[1].bar(model_names, position_errors, color=["#2563eb", "#94a3b8"])  # 比较解码后的物理位置误差
axes[1].set(title="Decoded object position error", ylabel="Absolute error")  # 标注可解释任务误差
for axis in axes:  # 统一旋转方法名称避免重叠
    axis.tick_params(axis="x", rotation=15)  # 轻微旋转两个模型名称
fig.tight_layout()  # 调整两幅柱状图间距
plt.show()  # 显示动作条件对潜表示和任务量的影响

**怎样理解结果：** 动作条件 latent 模型在表示 MSE 与解码位置误差上都更低；无动作模型主要学到训练动作均值接近零，因此倾向预测光点保持原位。第二个指标比像素或 latent MSE更直接回答“模型是否保留了用于位置规划的信息”。

## 3. 把 latent 预测反复作为下一步输入

从左侧位置开始，前 8 步向右移动，后 8 步向左移动。我们滚动动作条件与无动作 latent dynamics，并在几个时刻解码图像，检查低维预测怎样影响可见未来。

In [ ]:
rollout_actions = np.concatenate([np.full(8, 0.08), np.full(8, -0.06)])  # 建立先向右再向左的十六步动作序列
true_rollout_positions = [-0.6]  # 保存真实结构化位置轨迹
for action_value in rollout_actions:  # 逐步应用真实带边界位置动力学
    true_rollout_positions.append(float(np.clip(true_rollout_positions[-1] + action_value, -0.8, 0.8)))  # 根据动作更新并保存真实位置
true_rollout_positions = np.array(true_rollout_positions)  # 转换为便于绘图的位置数组
initial_image = render_position(true_rollout_positions[0])[None]  # 渲染 rollout 初始像素观测
action_latent = encode_images(initial_image)  # 初始化动作条件 World Model 的 latent 状态
state_latent = action_latent.copy()  # 初始化无动作 World Model 的相同 latent 状态
action_latent_trace = [action_latent[0].copy()]  # 保存动作条件 latent rollout
state_latent_trace = [state_latent[0].copy()]  # 保存无动作 latent rollout
for action_value in rollout_actions:  # 逐步使用两种模型自己的 latent 预测
    action_input = np.concatenate([action_latent, np.array([[action_value]]), np.ones((1, 1))], axis=1)  # 组织当前预测 latent、动作和偏置
    state_input = np.concatenate([state_latent, np.ones((1, 1))], axis=1)  # 组织忽略动作的当前预测 latent
    action_latent = action_latent + action_input @ action_weights  # 使用动作条件残差推进潜状态
    state_latent = state_latent + state_input @ state_weights  # 使用平均残差推进无动作潜状态
    action_latent_trace.append(action_latent[0].copy())  # 保存当前动作条件 latent
    state_latent_trace.append(state_latent[0].copy())  # 保存当前无动作 latent
action_rollout_images = decode_latents(np.stack(action_latent_trace))  # 解码动作条件的完整未来图像序列
state_rollout_images = decode_latents(np.stack(state_latent_trace))  # 解码无动作的完整未来图像序列
snapshot_steps = [0, 4, 8, 12, 16]  # 选择五个时间位置展示未来图像
fig, axes = plt.subplots(3, len(snapshot_steps), figsize=(10, 5.5))  # 创建真实、动作条件和无动作三行图像
for column_index, step_index in enumerate(snapshot_steps):  # 逐个时间位置填充三行观测
    axes[0, column_index].imshow(render_position(true_rollout_positions[step_index]), cmap="viridis", vmin=0.0, vmax=1.0)  # 显示真实未来像素
    axes[1, column_index].imshow(action_rollout_images[step_index], cmap="viridis", vmin=0.0, vmax=1.0)  # 显示动作条件 latent 解码未来
    axes[2, column_index].imshow(state_rollout_images[step_index], cmap="viridis", vmin=0.0, vmax=1.0)  # 显示忽略动作的 latent 解码未来
    axes[0, column_index].set_title(f"Step {step_index}")  # 标注当前未来时间位置
    for row_index in range(3):  # 统一隐藏当前列三幅图的像素刻度
        axes[row_index, column_index].axis("off")  # 只保留光点位置和形状
axes[0, 0].set_ylabel("True")  # 标记第一行为真实像素未来
axes[1, 0].set_ylabel("Latent + action")  # 标记第二行为动作条件潜空间未来
axes[2, 0].set_ylabel("Latent only")  # 标记第三行为忽略动作潜空间未来
fig.suptitle("Decoded latent rollout reveals what the dynamics preserved")  # 强调潜预测最终要回到可解释任务变化
fig.tight_layout()  # 调整十五幅小图间距
plt.show()  # 显示三种未来图像序列
fig, axis = plt.subplots(figsize=(8.8, 3.7))  # 创建三条物理位置轨迹对比图
axis.plot(true_rollout_positions, color="#172033", linewidth=2.5, label="True state")  # 绘制真实结构化位置
axis.plot(image_centers(action_rollout_images), color="#2563eb", label="Decoded latent + action")  # 绘制动作条件 latent 的解码位置
axis.plot(image_centers(state_rollout_images), color="#94a3b8", linestyle="--", label="Decoded latent only")  # 绘制无动作 latent 的解码位置
axis.set(title="Task-space measurement of the same latent rollout", xlabel="Rollout step", ylabel="Object position")  # 标注任务空间评价含义
axis.legend()  # 显示真实与两种模型轨迹图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察累积误差
fig.tight_layout()  # 调整位置曲线边距
plt.show()  # 显示 latent 未来在结构化状态中的误差

**怎样理解结果：** 动作条件 latent 能让光点先向右再向左，并在结构化位置曲线上大体跟随真实未来；忽略动作的模型把光点留在初始区域。解码图会出现 PCA 重建伪影，说明低维 latent 并不保存所有像素细节，但仍可保留任务需要的横向位置。

**本练习的结论：** 状态、像素和 latent 不是三套互斥算法，而是 World Model 的不同预测目标。应同时检查表示损失、解码观测和任务空间量，确认压缩没有丢掉规划所需信息。